### ЗАДАЧА: Пакетная обработка переводов между кошельками (exceptions + business rules)

Есть набор кошельков и список входящих переводов.
Нужно безопасно обработать пакет операций: валидные переводы применить к балансам,
ошибочные сохранить в отчёт, не останавливая всю обработку.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `TransferError`
   - `TransferFormatError`
   - `AccountNotFoundError`
   - `CurrencyMismatchError`
   - `InsufficientFundsError`
   - `TransferAmountError`.

2. Функцию `parse_transfer(raw)`:
   - формат строки: `transfer_id|from_user|to_user|amount`
   - `amount` должен быть числом и `> 0`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `apply_transfer(transfer, wallets)`:
   - проверить, что оба пользователя существуют
   - нельзя переводить самому себе
   - валюты кошельков отправителя и получателя должны совпадать
   - у отправителя должно хватать средств
   - при успехе обновить балансы в `wallets`
   - вернуть краткий словарь результата.

4. Функцию `process_batch(rows, wallets)`:
   - для каждой строки вызвать `parse_transfer`, потом `apply_transfer`
   - вернуть `(successes, errors)`
   - ошибки хранить как `(raw, error_type, message)`
   - не прерывать цикл на первой ошибке.

5. Вывести:
   - успешные переводы,
   - ошибки по типам,
   - итоговые балансы,
   - пользователя с максимальным балансом в валюте `USD`.


In [ ]:
wallets = {
    'alice': {'currency': 'USD', 'balance': 1200.0},
    'bob': {'currency': 'USD', 'balance': 450.0},
    'carol': {'currency': 'EUR', 'balance': 900.0},
    'dave': {'currency': 'USD', 'balance': 150.0},
}

rows = [
    'TR-100|alice|bob|200',
    'TR-101|bob|dave|700',
    'TR-102|alice|carol|50',
    'TR-103|eve|bob|30',
    'TR-104|dave|dave|10',
    'TR-105|bob|alice|abc',
    'TR-106|bob|dave|100',
]


class TransferError(Exception):
    pass


class TransferFormatError(TransferError):
    pass


class AccountNotFoundError(TransferError):
    pass


class CurrencyMismatchError(TransferError):
    pass


class InsufficientFundsError(TransferError):
    pass


class TransferAmountError(TransferError):
    pass


def parse_transfer(raw):
    # TODO: распарсить строку и вернуть dict перевода
    # TODO: при ошибке конвертации amount использовать raise ... from ...
    try:
        parts = raw.split('|')
        if len(parts) != 4:
            raise TransferFormatError(f"Некорректный формат строки: получено {len(parts)} вместо 4-х")

        transfer_id, from_user, to_user, amount = parts

        try:
            amount = float(amount)
        except ValueError as e:
            raise TransferAmountError("Некорректное значение суммы.") from e

        if amount <= 0:
            raise TransferAmountError("Сумма перевода должна быть > 0")

        return {
            'transfer_id': transfer_id,
            'from_user': from_user,
            'to_user': to_user,
            'amount': amount
        }
    except TransferError as e:
        raise e
    except Exception as e:
        raise TransferFormatError(f"Ошибка при обработке строки: {e}")


def apply_transfer(transfer, wallets):

    # TODO: проверить существование аккаунтов
    from_user = transfer['from_user']
    to_user = transfer['to_user']
    amount = transfer['amount']

    if from_user not in wallets:
        raise AccountNotFoundError(f"Отправитель не найден: {from_user}")
    if to_user not in wallets:
        raise AccountNotFoundError(f"Получатель не найден: {to_user}")

    # TODO: запретить перевод самому себе
    if from_user == to_user:
        raise TransferError(f"Нельзя переводить средства самому себе: {from_user}")

    from_wallet = wallets[from_user]
    to_wallet = wallets[to_user]

    # TODO: проверить совпадение валют
    if from_wallet['currency'] != to_wallet['currency']:
        raise CurrencyMismatchError(
            f"Несовпадение валют: {from_user} ({from_wallet['currency']}) → {to_user} ({to_wallet['currency']})"
        )
    # TODO: проверить баланс отправителя
    if from_wallet['balance'] < amount:
        raise InsufficientFundsError(
            f"Недостаточно средств: баланс {from_user} = {from_wallet['balance']}, требуется {amount}"
        )
    # TODO: обновить балансы и вернуть dict результата
    from_wallet['balance'] -= amount
    to_wallet['balance'] += amount

    return {
        'transfer_id': transfer['transfer_id'],
        'from_user': from_user,
        'to_user': to_user,
        'amount': amount,
        'currency': from_wallet['currency']
    }

def process_batch(rows, wallets):
# TODO: вернуть (successes, errors)
# TODO: вызвать process_batch(rows, wallets)
# TODO: вывести успешные переводы
    successes = []
    errors = []

    for raw in rows:
        try:
            transfer = parse_transfer(raw)
            result = apply_transfer(transfer, wallets)
            successes.append(result)
        except TransferError as e:
            errors.append((raw, type(e).__name__, str(e)))
        except Exception as e:
            errors.append((raw, 'UnknownError', str(e)))

    return successes, errors

wallets = {
    'alice': {'currency': 'USD', 'balance': 1200.0},
    'bob': {'currency': 'USD', 'balance': 450.0},
    'carol': {'currency': 'EUR', 'balance': 900.0},
    'dave': {'currency': 'USD', 'balance': 150.0},
}

rows = [
    'TR-100|alice|bob|200',
    'TR-101|bob|dave|700',
    'TR-102|alice|carol|50',
    'TR-103|eve|bob|30',
    'TR-104|dave|dave|10',
    'TR-105|bob|alice|abc',
    'TR-106|bob|dave|100',
]

successes, errors = process_batch(rows, wallets)

print("Успешные переводы:")
if successes:
    for success in successes:
        print(f"  {success['transfer_id']}: {success['from_user']} → {success['to_user']} | {success['amount']} {success['currency']}")
else:
    print("Нет успешных переводов")

# TODO: вывести ошибки по типам
error_counts = {}
for _, error_type, _ in errors:
    error_counts[error_type] = error_counts.get(error_type, 0) + 1

print("\nОшибки по типам:")
for error_type, count in error_counts.items():
    print(f"  {error_type}: {count}")

# TODO: вывести итоговые балансы
print("\nИтоговые балансы:")
for user, wallet in wallets.items():
    print(f"  {user}: {wallet['balance']} {wallet['currency']}")
    
# TODO: найти richest_usd_user
usd_users = {
    user: wallet['balance']
    for user, wallet in wallets.items()
    if wallet['currency'] == 'USD'
}

if usd_users:
    richest_usd_user = max(usd_users, key=usd_users.get)
    richest_usd_balance = usd_users[richest_usd_user]
    print(f"\nПользователь с максимальным балансом в USD: {richest_usd_user} ({richest_usd_balance} USD)")
else:
    print("\nНет пользователей с балансом в USD")

Успешные переводы:
  TR-100: alice → bob | 200.0 USD
  TR-106: bob → dave | 100.0 USD

Ошибки по типам:
  InsufficientFundsError: 1
  CurrencyMismatchError: 1
  AccountNotFoundError: 1
  TransferError: 1
  TransferAmountError: 1

Итоговые балансы:
  alice: 1000.0 USD
  bob: 550.0 USD
  carol: 900.0 EUR
  dave: 250.0 USD

Пользователь с максимальным балансом в USD: alice (1000.0 USD)
